# T1h — bases pequenas no lugar do MiniLM-L6

Gerado por `scripts/gerar_notebook_colab.py`; **não edite aqui** — o fonte é `colab/t1h_bases_pequenas.py`.

Código: commit `cb49b0d`.

**Antes de rodar:** Ambiente de execução → Alterar o tipo → **T4 GPU**, e suba `pares_treino.parquet` e `pares_validacao.parquet` para `/content/drive/MyDrive/phifm/pares`.


In [ ]:
# ─── 1. GPU, Drive, dependências, código e os pares ────────────────────────
import json, os, subprocess, sys, time
from pathlib import Path

import torch

# ⚠️ ABORTA sem GPU. A T4 do Colab não é garantida, e em CPU isto levaria dias.
assert torch.cuda.is_available(), (
    "sem GPU. Ambiente de execução → Alterar o tipo de ambiente de execução → T4 GPU.")
print(f"torch {torch.__version__} · {torch.cuda.get_device_name(0)} · "
      f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

from google.colab import drive
drive.mount("/content/drive")

PASTA = Path("/content/drive/MyDrive/phifm")
DADOS = PASTA / "pares"
SAIDA_BASE = PASTA / "runs" / "t1h"
SAIDA_BASE.mkdir(parents=True, exist_ok=True)
for nome in ("pares_treino.parquet", "pares_validacao.parquet"):
    assert (DADOS / nome).exists(), (
        f"{DADOS / nome} não existe. Os pares do pacote `kaggle_t2eq_emb` ficam "
        "nessa pasta do Drive — são os mesmos do passo 1 e do CPT.")

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "polars", "blake3", "tokenizers"], check=True)

# ── O código, do commit exato ──────────────────────────────────────────────
SHA = "cb49b0d8697ad0665db0aa48c1774ed573195771"
REPO = "sanchezVB/LLM_F-sica"
CODIGO = Path("/content/codigo")
if not CODIGO.exists():
    subprocess.run(["git", "clone", "--quiet", f"https://github.com/{REPO}.git",
                    str(CODIGO)], check=True)
subprocess.run(["git", "-C", str(CODIGO), "fetch", "--quiet", "--depth", "1",
                "origin", SHA], check=True)
subprocess.run(["git", "-C", str(CODIGO), "checkout", "--quiet", SHA], check=True)
FONTE = CODIGO / "src"
assert (CODIGO / "scripts/train_embedding.py").exists()
print(f"código em {CODIGO} · SHA {SHA[:7]}")

# ── Os pares TÊM de ser os mesmos bytes dos outros ajustes ─────────────────
from blake3 import blake3

HASHES = {
    "pares_treino.parquet": "6d4737e7d64e05ddad8071abbc83f98e1dece345ed5e0d3b9dfd871dcf63beb7",
    "pares_validacao.parquet": "a188bce65066fdc19aba30580188bf56711412a552316fd88d695157678084b4"
}
for nome, esperado in HASHES.items():
    h = blake3()
    with open(DADOS / nome, "rb") as f:
        while b := f.read(1 << 22):
            h.update(b)
    assert h.hexdigest() == esperado, (
        f"{nome}: blake3 {h.hexdigest()[:16]}… e o pacote declara {esperado[:16]}…\n"
        "Estes não são os pares dos outros ajustes, e os números deixariam de ser "
        "comparáveis.")
print(f"✅ pares conferidos por blake3 · saídas em {SAIDA_BASE}")

In [ ]:
# ─── 2. Treinar as cinco bases, uma depois da outra ────────────────────────
# ⚠️ Iguais aos do T1f e de todos os ajustes de 200 mil pares, conferidos por teste.
LOTE = 128
MAX_TOKENS = 192
SEMENTE = 17
N_CANDIDATOS = 1000
PASSOS_AVAL = 200
MAX_PARES = 200_000

BASES = {
    "minilm-l6": "sentence-transformers/all-MiniLM-L6-v2",
    "gte-small": "thenlper/gte-small",
    "bge-small": "BAAI/bge-small-en-v1.5",
    "e5-small": "intfloat/e5-small-v2",
    "minilm-l12": "sentence-transformers/all-MiniLM-L12-v2"
}

REGRA = r"""
  ── A REGRA, escrita ANTES (2026-09-23) ───────────────────────────────────

  A PERGUNTA: uma base moderna do porte do MiniLM-L6 (~33 M), ajustada na mesma
  receita, captura o ganho de base que o GTE-base mostrou — sem o custo de 4,4x?

  MEDIDA PRIMÁRIA: nDCG@10 no protocolo do G1, LOCAL, os cinco na mesma sessão.
  Cada candidata contra o MiniLM-L6@200k treinado AQUI, bootstrap pareado por
  ITEM. São 4 comparações contra o mesmo controle: correção de Bonferroni, IC de
  98,75% (0,05/4).

  Por candidata:
    ACIMA  (IC exclui 0, acima)  -> a base pesa também no porte pequeno.
    EMPATE (IC cruza 0)          -> no porte pequeno a base não se separa.
    ABAIXO (IC exclui 0, abaixo) -> o MiniLM-L6 já é a melhor base pequena.

  SECUNDÁRIA, e é ela que decide o PRODUTO: o custo de embutir, medido local no
  mesmo universo, em razão ao MiniLM-L6. Uma candidata ACIMA e com custo ≤ 2,5x
  vira candidata a subir de volume (1 M de pares, como o T1g) contra o ΦEmb do
  sistema. 2,5x porque o GTE-base, a 4,4x, só empatou — e uma base de 12 camadas
  com a largura do MiniLM custa cerca de 2x por FLOPs; acima de 2,5x a vantagem de
  custo que motiva este experimento deixa de existir.

  ⚠️ O e5-small e o bge-small foram feitos para usar prefixo, e aqui não usam — a
     desvantagem é declarada e igual para os dois.
  ⚠️ Uma semente por base, 200 mil pares — metade da receita do T1a.
  ⚠️ O nDCG que este notebook imprime é do pool INTERNO do treino. A comparação é
     LOCAL, pelo protocolo do G1.
"""
print(REGRA, flush=True)

ambiente = dict(os.environ, PYTHONPATH=str(FONTE), PYTHONIOENCODING="utf-8",
                TOKENIZERS_PARALLELISM="false")


def concluido(saida: Path) -> bool:
    """O treino grava `phiemb.json` com `completed_at` só quando termina."""
    meta = saida / "phiemb.json"
    if not meta.exists():
        return False
    return bool(json.loads(meta.read_text(encoding="utf-8")).get("completed_at"))


custos = {}
for nome, base in BASES.items():
    saida = SAIDA_BASE / f"phiemb-t1h-{nome}-200k"
    if concluido(saida) and (saida.parent / f"{saida.name}-melhor").exists():
        print(f"\n⏭️  {nome}: já concluído em {saida.name} — pulando", flush=True)
        continue
    log = SAIDA_BASE / f"treino_t1h_{nome}.log"
    cmd = [sys.executable, "-u", str(CODIGO / "scripts/train_embedding.py"),
           "--pares", str(DADOS), "--out", str(saida), "--base", base,
           "--lote", str(LOTE), "--max-tokens", str(MAX_TOKENS),
           "--max-pares", str(MAX_PARES), "--passos-aval", str(PASSOS_AVAL),
           "--n-candidatos", str(N_CANDIDATOS), "--dispositivo", "cuda",
           "--semente", str(SEMENTE)]
    print(f"\n{'=' * 74}\n▶ {nome} · {base}\n$ {' '.join(cmd)}", flush=True)
    t0 = time.perf_counter()
    with open(log, "a", encoding="utf-8") as fh:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, bufsize=1, encoding="utf-8",
                                errors="replace", env=ambiente, cwd=str(CODIGO))
        for linha in proc.stdout:
            fh.write(linha)
            fh.flush()
            print(linha, end="", flush=True)
        codigo_saida = proc.wait()
    custos[nome] = round(time.perf_counter() - t0, 1)
    assert codigo_saida == 0, (
        f"{nome} saiu com {codigo_saida}; log em {log}. Rodar esta célula de novo "
        "pula as bases concluídas e RETOMA esta do último checkpoint no Drive.")
    print(f"✅ {nome}: {custos[nome] / 60:.1f} min", flush=True)

In [ ]:
# ─── 3. O que baixar ───────────────────────────────────────────────────────
resumo = {"experimento": "t1h", "onde": "colab", "git_sha_codigo": SHA,
          "gpu": torch.cuda.get_device_name(0),
          "hiperparametros": {"lote": LOTE, "max_tokens": MAX_TOKENS,
                              "semente": SEMENTE, "max_pares": MAX_PARES,
                              "n_candidatos_aval": N_CANDIDATOS,
                              "passos_aval": PASSOS_AVAL},
          "custos_de_treino_s_desta_sessao": custos, "bases": {}, "regra": REGRA,
          "nota": ("nDCG do pool INTERNO do treino; a comparação é LOCAL, pelo "
                   "protocolo do G1, com o custo de embutir medido lá.")}
print("=" * 74)
for nome, base in BASES.items():
    melhor = SAIDA_BASE / f"phiemb-t1h-{nome}-200k-melhor"
    if not melhor.exists():
        print(f"  ❌ {nome}: sem checkpoint — rode a célula 2 de novo")
        continue
    m = json.loads((melhor / "melhor.json").read_text(encoding="utf-8"))
    resumo["bases"][nome] = {"base": base, "melhor": m}
    print(f"  {nome:<11} nDCG@10 interno {m.get('ndcg_10'):.4f} · passo {m.get('passo')}")
(SAIDA_BASE / "t1h.json").write_text(json.dumps(resumo, indent=2, ensure_ascii=False),
                                     encoding="utf-8")
print("=" * 74)
print(f"\nNo Drive, em {SAIDA_BASE}: as cinco pastas `*-melhor` e o `t1h.json` — é o")
print("que tem de vir para D:\\LLMFísica\\models\\. Pode baixar a pasta `t1h` inteira.")